# Day 11 — Mock Interview Practice

**Difficulty: INTERMEDIATE → ADVANCED** | **Week 2, Day 4**

**18 Questions across 8 Categories**

| # | Category | Topic | Level |
|---|----------|-------|-------|
| Q1 | SQL Advanced | CUME_DIST / PERCENT_RANK percentile analysis | INTERMEDIATE |
| Q2 | SQL Advanced | Conditional aggregation pivot + YoY growth | ADVANCED |
| Q3 | Python | Trapping Rain Water (two-pointer) | INTERMEDIATE |
| Q4 | Python | Trie with autocomplete & prefix search | ADVANCED |
| Q5 | PySpark Coding | Multi-level groupBy + top-N window ranking | INTERMEDIATE |
| Q6 | PySpark Coding | Schema evolution with mergeSchema & nested structs | ADVANCED |
| Q7 | Spark Optimization | Shuffle partitioning & tuning internals | INTERMEDIATE |
| Q8 | Spark Optimization | Dynamic resource allocation & executor lifecycle | ADVANCED |
| Q9 | ETL Fault Tolerance | Dead letter queue: quarantine + replay | INTERMEDIATE |
| Q10 | ETL Fault Tolerance | Fault tolerance: retries + Repair Runs + checkpoint | ADVANCED |
| Q11 | ETL Fault Tolerance | Idempotent batch processing with run-id ledger | ADVANCED |
| Q12 | Data Modeling | Junk & degenerate dimensions | INTERMEDIATE |
| Q13 | Data Modeling | Surrogate key strategies with MERGE SCD Type 2 | ADVANCED |
| Q14 | Snowflake | Snowpipe vs Snowpipe Streaming ingestion | INTERMEDIATE |
| Q15 | Snowflake | Task DAG with serverless tasks orchestration | ADVANCED |
| Q16 | Monitoring & Orchestration | Jobs retry & timeout — resilient pipelines | INTERMEDIATE |
| Q17 | Monitoring & Orchestration | Unified observability with system tables | ADVANCED |
| Q18 | Monitoring & Orchestration | Self-healing pipeline: detect + retry + DLQ + alert | ADVANCED |

## Q1: CUME_DIST & PERCENT_RANK — Salary Percentile Analysis [INTERMEDIATE]

**Category: SQL Advanced**

Given an `employees` table with columns `(emp_id, emp_name, department, salary)`:

1. For each department, compute each employee's **PERCENT_RANK** (relative rank: 0 = lowest, 1 = highest)
2. Compute **CUME_DIST** (cumulative distribution) — fraction of employees in the same department earning <= that employee's salary
3. Use **NTILE(4)** to bucket each department's salaries into quartiles (quartile 1 = highest earners)
4. Identify all employees in the **top quartile** (quartile = 1) across all departments

**Requirements:**
- Use Databricks SQL window functions
- Create sample data with `spark.createDataFrame()`
- Explain the difference between PERCENT_RANK and CUME_DIST

**Hint:** PERCENT_RANK = (rank - 1) / (total_rows - 1); CUME_DIST = number of rows <= current / total_rows

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# --- Sample Data ---
data = [
    (1, "Alice",   "Engineering", 120000),
    (2, "Bob",     "Engineering",  95000),
    (3, "Charlie", "Engineering", 110000),
    (4, "Diana",   "Engineering",  80000),
    (5, "Eve",     "Sales",        70000),
    (6, "Frank",   "Sales",        85000),
    (7, "Grace",   "Sales",        60000),
    (8, "Heidi",   "Marketing",    90000),
    (9, "Ivan",    "Marketing",    75000),
    (10, "Judy",   "Marketing",   100000),
]

df = spark.createDataFrame(data, ["emp_id", "emp_name", "department", "salary"])
df.createOrReplaceTempView("employees")

# --- SQL Solution ---
result = spark.sql("""
WITH percentiles AS (
  SELECT
    emp_id, emp_name, department, salary,
    PERCENT_RANK() OVER (PARTITION BY department ORDER BY salary) AS pct_rank,
    CUME_DIST()   OVER (PARTITION BY department ORDER BY salary) AS cume_dist,
    NTILE(4)      OVER (PARTITION BY department ORDER BY salary DESC) AS quartile
  FROM employees
)
SELECT * FROM percentiles
ORDER BY department, salary DESC
""")
result.show(truncate=False)

# --- Top Quartile Employees ---
top_q = spark.sql("""
WITH q AS (
  SELECT emp_name, department, salary,
         NTILE(4) OVER (PARTITION BY department ORDER BY salary DESC) AS quartile
  FROM employees
)
SELECT emp_name, department, salary
FROM q
WHERE quartile = 1
ORDER BY salary DESC
""")
print("Top-Quartile Employees:")
top_q.show(truncate=False)

# --- Explanation ---
print("""
PERCENT_RANK vs CUME_DIST:
  - PERCENT_RANK: relative position in the ordered set. Formula: (rank-1)/(N-1).
    Range: [0, 1].  First row = 0, last row = 1.  Gap-aware.
  - CUME_DIST: cumulative distribution = (# rows with value <= current) / N.
    Range: (0, 1].  Multiple rows with the same value share the same CUME_DIST.
  - Key diff: PERCENT_RANK penalises gaps; CUME_DIST treats ties equally.
""")

# --- Assertions ---
assert result.count() == 10
assert top_q.count() >= 3   # at least one per department top quartile
print("Q1 PASSED")

## Q2: Conditional Aggregation Pivot + YoY Growth [ADVANCED]

**Category: SQL Advanced**

Given a `sales` table with columns `(sale_date, category, amount)` spanning 2024-2025:

1. **Pivot** categories into columns using **conditional aggregation** (`SUM(CASE WHEN ...)`), producing monthly total sales per category per year-month
2. Compute **Year-over-Year (YoY) growth percentage** for each category-month combination (e.g., Jan 2025 vs Jan 2024)
3. Demonstrate why conditional aggregation is preferred over PIVOT for dynamic column sets

**Requirements:**
- Use Databricks SQL with `YEAR()` and `MONTH()` functions
- Create sample data with `spark.createDataFrame()` using `datetime.date` objects
- Handle NULLs for months without prior-year data

**Hint:** Use `LAG(monthly_total) OVER (PARTITION BY category, sale_month ORDER BY sale_year)` for YoY comparison.

In [0]:
from datetime import date

# --- Sample Data (2024 + 2025) ---
data = [
    (date(2024, 1, 15), "Electronics", 5000),
    (date(2024, 1, 20), "Clothing",    3000),
    (date(2024, 2, 10), "Electronics", 4500),
    (date(2024, 2, 15), "Clothing",    2800),
    (date(2024, 3,  5), "Electronics", 6000),
    (date(2024, 3, 12), "Clothing",    3200),
    (date(2025, 1,  8), "Electronics", 7000),
    (date(2025, 1, 18), "Clothing",    3500),
    (date(2025, 2, 10), "Electronics", 5200),
    (date(2025, 2, 20), "Clothing",    3100),
    (date(2025, 3,  5), "Electronics", 7500),
    (date(2025, 3, 15), "Clothing",    3800),
]

df = spark.createDataFrame(data, ["sale_date", "category", "amount"])
df.createOrReplaceTempView("sales")

# --- Conditional Aggregation Pivot ---
pivot_df = spark.sql("""
SELECT
  YEAR(sale_date)  AS sale_year,
  MONTH(sale_date) AS sale_month,
  SUM(CASE WHEN category = 'Electronics' THEN amount ELSE 0 END) AS electronics_total,
  SUM(CASE WHEN category = 'Clothing'    THEN amount ELSE 0 END) AS clothing_total,
  SUM(amount) AS grand_total
FROM sales
GROUP BY YEAR(sale_date), MONTH(sale_date)
ORDER BY sale_year, sale_month
""")
print("Monthly Pivot (Conditional Aggregation):")
pivot_df.show()

# --- YoY Growth ---
yoy_df = spark.sql("""
WITH monthly AS (
  SELECT
    YEAR(sale_date)  AS sale_year,
    MONTH(sale_date) AS sale_month,
    category,
    SUM(amount) AS monthly_total
  FROM sales
  GROUP BY YEAR(sale_date), MONTH(sale_date), category
),
yoy AS (
  SELECT
    sale_year, sale_month, category, monthly_total,
    LAG(monthly_total) OVER (PARTITION BY category, sale_month ORDER BY sale_year) AS prev_year_total
  FROM monthly
)
SELECT
  sale_year, sale_month, category,
  monthly_total, prev_year_total,
  CASE WHEN prev_year_total IS NOT NULL AND prev_year_total > 0
       THEN ROUND((monthly_total - prev_year_total) * 100.0 / prev_year_total, 2)
       ELSE NULL
  END AS yoy_growth_pct
FROM yoy
WHERE prev_year_total IS NOT NULL
ORDER BY category, sale_month, sale_year
""")
print("Year-over-Year Growth %:")
yoy_df.show()

print("""
Why conditional aggregation over PIVOT?
  1. PIVOT requires hard-coded column values — not suitable when categories are dynamic.
  2. Conditional aggregation lets you mix multiple measures in one query.
  3. Easier to add WHERE filters inside CASE WHEN for sub-categorisation.
  4. PIVOT can't do YoY in the same pass; conditional aggregation + LAG can.
""")

# --- Assertions ---
assert pivot_df.count() == 6   # 3 months x 2 years
assert yoy_df.count() == 6      # 2 categories x 3 months (2025 with 2024 prev)
print("Q2 PASSED")

## Q3: Trapping Rain Water — Two-Pointer Technique [INTERMEDIATE]

**Category: Python — Loop/Function/List/Dictionary**

Given an array of non-negative integers representing an elevation map where the width of each bar is 1, compute how much water it can trap after raining.

**Example:** `[0,1,0,2,1,0,1,3,2,1,2,1]` → **6**

**Visual:**
```
       █
   █───███
 █─████████
```

**Requirements:**
- Solve in O(n) time and O(1) space using the two-pointer technique
- Explain why two-pointer works (water level determined by the shorter side)
- Include test cases

**Hint:** Use left and right pointers. At each step, process the side with the lower max barrier. Water trapped = max_barrier - current_height.

In [0]:
def trap_rain_water(height):
    """Two-pointer: O(n) time, O(1) space."""
    if not height:
        return 0

    left, right = 0, len(height) - 1
    left_max, right_max = height[left], height[right]
    water = 0

    while left < right:
        if left_max < right_max:
            left += 1
            left_max = max(left_max, height[left])
            water += left_max - height[left]
        else:
            right -= 1
            right_max = max(right_max, height[right])
            water += right_max - height[right]

    return water

# --- Test Cases ---
assert trap_rain_water([0,1,0,2,1,0,1,3,2,1,2,1]) == 6
assert trap_rain_water([4,2,0,3,2,5]) == 9
assert trap_rain_water([1,0,1]) == 1
assert trap_rain_water([5,4,3,2,1]) == 0   # decreasing, no water
assert trap_rain_water([1,2,3,4,5]) == 0   # increasing, no water
assert trap_rain_water([3,0,2,0,4]) == 7
assert trap_rain_water([]) == 0
assert trap_rain_water([1]) == 0

print("All test cases passed!")
print(f"Example [0,1,0,2,1,0,1,3,2,1,2,1] -> {trap_rain_water([0,1,0,2,1,0,1,3,2,1,2,1])}")

# --- Explanation ---
print("""
Why two-pointer works:
  - Water level at any position is determined by min(left_max, right_max).
  - If left_max < right_max, the water at 'left' position depends ONLY on left_max
    (the right side is guaranteed to have a higher barrier).
  - So we can safely process the shorter side without knowing the exact right_max.
  - This avoids the O(n) space of the prefix/suffix max arrays.
""")
print("Q3 PASSED")

## Q4: Trie with Autocomplete & Prefix Search [ADVANCED]

**Category: Python — Loop/Function/List/Dictionary**

Implement a **Trie (prefix tree)** that supports:

1. `insert(word)` — insert a word into the trie
2. `search(word)` — return True if the exact word exists
3. `starts_with(prefix)` — return True if any word starts with the given prefix
4. `auto_complete(prefix)` — return all words with the given prefix (sorted)

**Requirements:**
- Use a TrieNode class with a `children` dict and `is_end` flag
- `auto_complete` should use DFS from the prefix node to collect all completions
- Include test cases demonstrating all four operations

**Example:**
```
trie.insert("apple"); trie.insert("app"); trie.insert("apt")
trie.search("app") -> True
trie.starts_with("ap") -> True
trie.auto_complete("ap") -> ["app", "apple", "apt"]
```

**Hint:** For auto_complete, navigate to the prefix node, then DFS collecting all paths where is_end=True.

In [0]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_end = False

class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word):
        node = self.root
        for ch in word:
            if ch not in node.children:
                node.children[ch] = TrieNode()
            node = node.children[ch]
        node.is_end = True

    def search(self, word):
        node = self._find_node(word)
        return node is not None and node.is_end

    def starts_with(self, prefix):
        return self._find_node(prefix) is not None

    def auto_complete(self, prefix):
        node = self._find_node(prefix)
        if node is None:
            return []
        results = []
        self._dfs(node, prefix, results)
        return sorted(results)

    def _find_node(self, prefix):
        node = self.root
        for ch in prefix:
            if ch not in node.children:
                return None
            node = node.children[ch]
        return node

    def _dfs(self, node, path, results):
        if node.is_end:
            results.append(path)
        for ch, child in sorted(node.children.items()):
            self._dfs(child, path + ch, results)

# --- Test Cases ---
trie = Trie()
trie.insert("apple")
trie.insert("app")
trie.insert("apt")
trie.insert("banana")
trie.insert("band")
trie.insert("bandana")

assert trie.search("app") == True
assert trie.search("apple") == True
assert trie.search("ap") == False       # prefix but not a complete word
assert trie.search("appl") == False
assert trie.search("bandana") == True
assert trie.search("cat") == False

assert trie.starts_with("ap") == True
assert trie.starts_with("ban") == True
assert trie.starts_with("cat") == False
assert trie.starts_with("bandana") == True

assert trie.auto_complete("ap") == ["app", "apple", "apt"]
assert trie.auto_complete("ban") == ["banana", "band", "bandana"]
assert trie.auto_complete("band") == ["band", "bandana"]
assert trie.auto_complete("z") == []
assert trie.auto_complete("") == ["app", "apple", "apt", "banana", "band", "bandana"]

print("All trie tests passed!")
print(f"auto_complete('ap') -> {trie.auto_complete('ap')}")
print(f"auto_complete('ban') -> {trie.auto_complete('ban')}")
print("Q4 PASSED")

## Q5: Multi-Level GroupBy + Conditional Aggregation + Top-N Window [INTERMEDIATE]

**Category: PySpark Coding**

Given a `transactions` table with columns `(txn_id, customer_id, category, amount, txn_date)`:

1. Compute **total spending** per customer per category using `groupBy`
2. Rank customers within each category by total spending and select the **top 3** per category using a window function
3. Add a conditional flag: `"high_value"` if amount > 500, `"normal"` otherwise, and compute the **count of high-value vs normal** transactions per category

**Requirements:**
- Use PySpark DataFrame API (not raw SQL)
- Use `Window.partitionBy()` for ranking
- Create sample data with `spark.createDataFrame()`
- Verify results with assertions

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
from datetime import date

# --- Sample Data ---
schema = StructType([
    StructField("txn_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("category", StringType(), True),
    StructField("amount", IntegerType(), True),
    StructField("txn_date", DateType(), True),
])

data = [
    (1, 101, "Electronics", 800, date(2025, 1, 15)),
    (2, 102, "Electronics", 300, date(2025, 1, 20)),
    (3, 101, "Electronics", 600, date(2025, 2, 10)),
    (4, 103, "Electronics", 900, date(2025, 2, 15)),
    (5, 104, "Electronics", 200, date(2025, 3, 5)),
    (6, 101, "Clothing",    450, date(2025, 1, 18)),
    (7, 102, "Clothing",    700, date(2025, 2, 12)),
    (8, 103, "Clothing",    250, date(2025, 3, 1)),
    (9, 104, "Clothing",    550, date(2025, 3, 10)),
    (10, 101, "Books",      150, date(2025, 1, 5)),
    (11, 102, "Books",      650, date(2025, 2, 8)),
    (12, 103, "Books",      400, date(2025, 3, 15)),
]

df = spark.createDataFrame(data, schema)

# --- 1. Total spending per customer per category ---
customer_cat = (df.groupBy("customer_id", "category")
    .agg(F.sum("amount").alias("total_spending"))
    .orderBy("category", F.desc("total_spending")))
print("Total spending per customer per category:")
customer_cat.show()

# --- 2. Top 3 customers per category by total spending ---
rank_window = Window.partitionBy("category").orderBy(F.desc("total_spending"))
top3 = (customer_cat
    .withColumn("rank", F.rank().over(rank_window))
    .filter(F.col("rank") <= 3)
    .orderBy("category", "rank"))
print("Top 3 Customers per Category:")
top3.show()

# --- 3. High-value vs normal transaction counts per category ---
flagged = df.withColumn("value_flag", F.when(F.col("amount") > 500, "high_value").otherwise("normal"))
flag_summary = (flagged.groupBy("category", "value_flag")
    .agg(F.count("*").alias("txn_count"))
    .orderBy("category", "value_flag"))
print("High-Value vs Normal per Category:")
flag_summary.show()

# --- Assertions ---
assert customer_cat.count() == 9   # 4 customers x 3 categories minus 1 missing combo (104 in Books)
assert top3.filter(F.col("category") == "Electronics").count() == 3
assert flag_summary.count() == 6    # 3 categories x 2 flags
print("Q5 PASSED")

## Q6: Schema Evolution with mergeSchema & Nested Struct Changes [ADVANCED]

**Category: PySpark Coding**

You have a Delta table with nested struct columns. New data arrives with **additional fields** inside the struct:

1. Create an initial Delta table with a nested `address` struct containing `street` and `city`
2. Append new data where the `address` struct has an additional `zip_code` field, using `mergeSchema`
3. Query the evolved schema and demonstrate how **old rows** get NULL for the new field
4. Show how `autoMerge` works with `MERGE INTO` for schema evolution

**Requirements:**
- Use UC-managed Delta tables (no DBFS paths)
- Demonstrate `mergeSchema` option on write
- Show the schema before and after evolution
- Verify old rows have NULL for new fields

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# --- 1. Initial Delta table with nested struct ---
initial_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("address", StructType([
        StructField("street", StringType(), True),
        StructField("city", StringType(), True),
    ]), True),
])

initial_data = [
    (1, "Alice", {"street": "123 Main St", "city": "NYC"}),
    (2, "Bob",   {"street": "456 Oak Ave", "city": "LA"}),
]

df_initial = spark.createDataFrame(initial_data, initial_schema)
tbl_name = "workspace.default.day11_q6_schema_evol"
df_initial.write.mode("overwrite").saveAsTable(tbl_name)

print("=== Initial Schema ===")
spark.table(tbl_name).printSchema()
spark.table(tbl_name).show(truncate=False)

# --- 2. Append new data with additional 'zip_code' field using mergeSchema ---
evolved_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("address", StructType([
        StructField("street", StringType(), True),
        StructField("city", StringType(), True),
        StructField("zip_code", StringType(), True),  # NEW field
    ]), True),
])

new_data = [
    (3, "Charlie", {"street": "789 Pine Rd", "city": "Chicago", "zip_code": "60601"}),
    (4, "Diana",   {"street": "321 Elm St", "city": "Boston",  "zip_code": "02108"}),
]

df_new = spark.createDataFrame(new_data, evolved_schema)
df_new.write.mode("append").option("mergeSchema", "true").saveAsTable(tbl_name)

print("=== Evolved Schema (after mergeSchema) ===")
spark.table(tbl_name).printSchema()
spark.table(tbl_name).show(truncate=False)

# --- 3. Verify old rows have NULL for zip_code ---
result = spark.table(tbl_name)
old_rows = result.filter(F.col("id").isin([1, 2]))
new_rows = result.filter(F.col("id").isin([3, 4]))

old_zip = old_rows.select("id", F.col("address.zip_code").alias("zip_code")).collect()
new_zip = new_rows.select("id", F.col("address.zip_code").alias("zip_code")).collect()

print("Old rows zip_code (should be NULL):")
for r in old_zip:
    print(f"  id={r['id']}, zip_code={r['zip_code']}")
    assert r['zip_code'] is None

print("New rows zip_code (should have values):")
for r in new_zip:
    print(f"  id={r['id']}, zip_code={r['zip_code']}")
    assert r['zip_code'] is not None

# --- 4. MERGE INTO with autoMerge ---
update_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("address", StructType([
        StructField("street", StringType(), True),
        StructField("city", StringType(), True),
        StructField("zip_code", StringType(), True),
        StructField("country", StringType(), True),  # ANOTHER new field
    ]), True),
])

merge_data = [
    (1, "Alice", {"street": "123 Main St", "city": "NYC", "zip_code": "10001", "country": "USA"}),
    (5, "Eve",   {"street": "999 Birch Ln", "city": "Seattle", "zip_code": "98101", "country": "USA"}),
]

df_merge = spark.createDataFrame(merge_data, update_schema)
df_merge.createOrReplaceTempView("updates_v")

spark.sql(f"""
MERGE INTO {tbl_name} AS t
USING updates_v AS s
ON t.id = s.id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

print("=== After MERGE with autoMerge ===")
spark.table(tbl_name).printSchema()
spark.table(tbl_name).orderBy("id").show(truncate=False)

assert spark.table(tbl_name).count() == 5
assert "country" in [f.name for f in spark.table(tbl_name).schema.fields]
print("Q6 PASSED")

## Q7: Shuffle Partitioning & Tuning Internals [INTERMEDIATE]

**Category: Spark Optimization & Deep Spark**

Explain and demonstrate Spark shuffle mechanics:

1. What causes a **shuffle** in Spark? List the common operations (groupBy, join, repartition, distinct, orderBy)
2. What is `spark.sql.shuffle.partitions` and why is the default (200) often suboptimal?
3. How do you choose the right number of shuffle partitions? Give a formula.
4. Demonstrate **data skew** in a groupBy and how it affects partition distribution
5. How does **AQE coalescing** help post-shuffle?

**Requirements:**
- Show partition sizes before and after shuffle
- Explain the formula: partitions ~ max(total_shuffle_bytes / target_partition_size, 1)
- Mention serverless AQE behavior (configs are managed)

In [0]:
from pyspark.sql import functions as F

# --- Demo: Shuffle caused by groupBy ---
data = [(f"key_{i % 3}", i) for i in range(1000)]  # 3 keys, skewed
df = spark.createDataFrame(data, ["key", "value"])

# Show partition count before shuffle
print(f"Default partitions (parallelism): {df.rdd.getNumPartitions() if hasattr(df, 'rdd') else 'N/A on serverless'}")

# groupBy triggers a shuffle -- check resulting partition count
grouped = df.groupBy("key").agg(F.count("*").alias("cnt"))
print(f"Partitions after groupBy (shuffle): {grouped.rdd.getNumPartitions() if hasattr(grouped, 'rdd') else 'N/A on serverless'}")
grouped.show()

# --- Explanation ---
print("""
SHUFFLE PARTITIONING EXPLAINED:
================================

1. What causes a shuffle?
   - Wide transformations: groupBy, reduceByKey, join, repartition, distinct, orderBy, window functions
   - Data must be redistributed across the network so that all records for a given key
     end up on the same partition (partitioner re-distribution).
   - Each shuffle stage writes intermediate data to disk (sort-based shuffle).

2. spark.sql.shuffle.partitions:
   - Default = 200 (set for small clusters; often too many for small data, too few for large data).
   - Each partition is processed by a single task. Too many partitions = overhead of tiny tasks.
   - Too few partitions = OOM or long task times.

3. Choosing the right number:
   Formula: partitions = max(total_shuffle_bytes / target_partition_size, 1)
   - target_partition_size: 128 MB (general purpose) or 256 MB (large scale).
   - Example: 50 GB shuffle / 128 MB = ~400 partitions.
   - Rule of thumb: partitions ~ 2-4x number of cores in cluster.
   - On serverless: AQE handles this automatically (spark.sql.adaptive.coalescePartitions.enabled=true).

4. Data skew:
   - If one key has 80% of the data, one task processes 80% while others sit idle.
   - Symptom: one task takes much longer than others (visible in Spark UI stage details).
   - Fix: salting (add random prefix), broadcast join for small dim, or AQE skew splitting.

5. AQE coalescing:
   - After the shuffle, AQE looks at actual partition sizes and merges small ones.
   - spark.sql.adaptive.coalescePartitions.enabled = true (default on Databricks).
   - On serverless: AQE configs are MANAGED -- cannot be set manually.
     The platform auto-tunes based on workload.
""")

# --- Demonstrate skew awareness ---
sizes = df.groupBy("key").count().orderBy(F.desc("count")).collect()
print("Key distribution (skew check):")
for r in sizes:
    print(f"  {r['key']}: {r['count']} rows")

# AQE config check (serverless-managed)
try:
    aqe = spark.conf.get("spark.sql.adaptive.enabled")
    print(f"\nAQE enabled: {aqe}")
except Exception:
    print("\nAQE config is managed on serverless (cannot read directly)")

print("Q7 PASSED")

## Q8: Dynamic Resource Allocation & Executor Lifecycle [ADVANCED]

**Category: Spark Optimization & Deep Spark**

Explain **Dynamic Resource Allocation (DRA)** in Spark:

1. What problem does DRA solve? Why is static allocation suboptimal?
2. Explain the **executor lifecycle**: request -> allocate -> idle -> deallocate
3. Key configs: `spark.dynamicAllocation.enabled`, `minExecutors`, `maxExecutors`, `initialExecutors`, `executorIdleTimeout`
4. What is **shuffle tracking** (`spark.dynamicAllocation.shuffleTracking.enabled`) and why was it introduced?
5. How does DRA interact with **structured streaming** and **continuous workloads**?
6. On Databricks serverless, how is resource allocation managed differently?

**Requirements:**
- Provide config examples for a classic Spark cluster
- Explain the difference between DRA on classic clusters vs serverless
- Draw a timeline of executor allocation/deallocation

In [0]:
print("""
DYNAMIC RESOURCE ALLOCATION (DRA) EXPLAINED:
=============================================

1. Problem DRA solves:
   - Static allocation: you request N executors upfront. If the workload needs fewer,
     resources are wasted. If it needs more, the job slows down or fails.
   - DRA: Spark dynamically requests and releases executors based on pending tasks.
   - Benefit: cost savings (release idle executors) + elastic scaling (request more under load).

2. Executor Lifecycle:
   Timeline:
   [Request] --> [Allocate] --> [Running (processing tasks)] --> [Idle] --> [IdleTimeout] --> [Deallocate]

   - Request: Driver asks the cluster manager for an executor when pending tasks > 0.
   - Allocate: Cluster manager assigns a container with the requested resources.
   - Running: Executor processes assigned tasks. Heartbeat to driver every 10s.
   - Idle: No tasks assigned. Timer starts.
   - IdleTimeout (default 60s): If no tasks for this duration, executor is removed.
   - Deallocate: Executor is killed. But shuffle files must be preserved (see #4).

3. Key Configurations (classic Spark):
   spark.dynamicAllocation.enabled = true
   spark.dynamicAllocation.minExecutors = 0       # Can scale to zero
   spark.dynamicAllocation.maxExecutors = 20      # Upper bound
   spark.dynamicAllocation.initialExecutors = 2   # Start with 2
   spark.dynamicAllocation.executorIdleTimeout = 60s  # Kill after 60s idle
   spark.dynamicAllocation.schedulerBacklogTimeout = 1s  # Request if backlog > 1s
   spark.dynamicAllocation.cachedExecutorIdleTimeout = 600s  # Keep cached longer

   On Databricks classic clusters, DRA is enabled by default with auto-tuned values.

4. Shuffle Tracking:
   spark.dynamicAllocation.shuffleTracking.enabled = true

   - BEFORE shuffle tracking: executors with shuffle output files could NOT be removed
     (other stages might need to fetch shuffle blocks from them).
   - Workaround: external shuffle service (ESS) had to be running on every node.
   - WITH shuffle tracking: the driver tracks which shuffle blocks are needed.
     It keeps executors alive until all dependent stages have fetched their blocks,
     THEN releases them. No ESS needed!
   - Introduced in Spark 3.0, production-ready in 3.2+.

5. DRA with Structured Streaming:
   - Streaming has continuous micro-batches. Between batches, executors go idle.
   - DRA might release executors between batches, causing re-allocation latency.
   - Solution: set minExecutors > 0 to always keep a baseline.
   - Or: use spark.dynamicAllocation.cachedExecutorIdleTimeout with a longer value.

6. Databricks Serverless:
   - DRA is NOT user-configurable on serverless compute.
   - The platform auto-scales based on query demand and data size.
   - No cluster management overhead -- you only pay for compute used.
   - Equivalent to DRA with intelligent defaults, but managed by the platform.
   - Serverless uses 'serverless compute' which is shared and elastically provisioned.

CONFIG EXAMPLE (classic cluster - for reference):

  spark-submit \\
    --conf spark.dynamicAllocation.enabled=true \\
    --conf spark.dynamicAllocation.minExecutors=1 \\
    --conf spark.dynamicAllocation.maxExecutors=20 \\
    --conf spark.dynamicAllocation.initialExecutors=2 \\
    --conf spark.dynamicAllocation.executorIdleTimeout=60s \\
    --conf spark.dynamicAllocation.shuffleTracking.enabled=true
""")
print("Q8 PASSED")

## Q9: Dead Letter Queue Pattern — Quarantine + Replay [INTERMEDIATE]

**Category: ETL — Failures, Late Data, Schema Handling, Fault Tolerance**

Design a **Dead Letter Queue (DLQ)** pattern for a batch ingestion pipeline:

1. You receive JSON records, some of which are malformed or fail validation (e.g., missing required fields, negative amounts)
2. **Valid** records should be written to the main `orders` Delta table
3. **Invalid** records should be routed to a `orders_dlq` quarantine table with an `error_reason` column
4. After fixing the root cause, demonstrate **replaying** quarantined records back into the main table

**Requirements:**
- Use PySpark with Delta Lake
- Use `foreachBatch` or a two-step approach to split good/bad records
- Include validation logic: `order_id` not null, `amount` > 0, `customer_id` not null
- Verify the replay works correctly

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

# --- Sample raw data with some bad records ---
raw_data = [
    (1, "C001", 150.00,   None),          # valid
    (2, "C002", -50.00,   None),          # invalid: negative amount
    (3, None,  300.00,    None),          # invalid: null customer_id
    (4, "C003", 75.50,    None),          # valid
    (5, "C004", 0.00,     None),          # invalid: zero amount
    (None, "C005", 200.00, None),         # invalid: null order_id
    (6, "C006", 450.00,   None),          # valid
]

schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("error_reason", StringType(), True),
])

df_raw = spark.createDataFrame(raw_data, schema)

main_tbl = "workspace.default.day11_q9_orders"
dlq_tbl = "workspace.default.day11_q9_orders_dlq"

# --- Validation: split good vs bad records ---
valid_df = df_raw.filter(
    F.col("order_id").isNotNull() &
    F.col("customer_id").isNotNull() &
    (F.col("amount") > 0)
)

invalid_df = (df_raw.filter(
    F.col("order_id").isNull() |
    F.col("customer_id").isNull() |
    (F.col("amount") <= 0)
).withColumn("error_reason",
    F.when(F.col("order_id").isNull(), "null_order_id")
     .when(F.col("customer_id").isNull(), "null_customer_id")
     .when(F.col("amount") <= 0, "invalid_amount")
))

# --- Write to main table and DLQ ---
valid_df.drop("error_reason").write.mode("overwrite").saveAsTable(main_tbl)
invalid_df.write.mode("overwrite").saveAsTable(dlq_tbl)

print("=== Valid Orders (main table) ===")
spark.table(main_tbl).show()
print(f"Valid count: {spark.table(main_tbl).count()}")

print("=== Dead Letter Queue (quarantine) ===")
spark.table(dlq_tbl).show()
print(f"DLQ count: {spark.table(dlq_tbl).count()}")

# --- Replay: fix the bad records and move back to main table ---
# Fix: set negative/zero amounts to abs, fill nulls
repaired = (spark.table(dlq_tbl)
    .filter(F.col("error_reason") == "invalid_amount")
    .withColumn("amount", F.abs(F.col("amount")))
    .filter(F.col("amount") > 0)  # still valid after fix
    .drop("error_reason"))

# Append repaired records to main table
if repaired.count() > 0:
    repaired.write.mode("append").saveAsTable(main_tbl)
    # Remove repaired records from DLQ
    spark.sql(f"DELETE FROM {dlq_tbl} WHERE error_reason = 'invalid_amount'")

print("=== After Replay ===")
print(f"Main table count: {spark.table(main_tbl).count()}")
print(f"DLQ count: {spark.table(dlq_tbl).count()}")
spark.table(main_tbl).orderBy("order_id").show()
spark.table(dlq_tbl).show()

# --- Assertions ---
assert spark.table(main_tbl).count() == 4  # 3 valid + 1 repaired (order 2)
assert spark.table(dlq_tbl).count() == 2    # remaining bad: null order_id, null customer_id
print("Q9 PASSED")

## Q10: Fault Tolerance — Task Retries + Repair Runs + Checkpoint Recovery [ADVANCED]

**Category: ETL — Failures, Late Data, Schema Handling, Fault Tolerance**

Design a **fault-tolerant ETL pipeline** on Databricks that handles failures at multiple levels:

1. **Task-level retries**: Configure `max_retries` and `min_retry_interval_millis` for individual tasks in a Databricks Job
2. **Job-level retry with Repair Runs**: When a multi-task job fails mid-way, use Repair Run to re-run only failed tasks (not the entire job)
3. **Checkpoint-based recovery** for streaming: if a structured streaming query crashes, it resumes from the last committed checkpoint offset
4. **Idempotent re-runs**: Design MERGE logic so that re-running a failed batch does not produce duplicates

**Requirements:**
- Provide Databricks Jobs JSON config for task retries
- Explain the Repair Run flow and API call
- Show streaming checkpoint configuration
- Provide MERGE-based idempotent write pattern with a run_id column
- Explain when to use each fault tolerance mechanism

In [0]:
print("""
FAULT TOLERANCE IN DATABRICKS ETL PIPELINES:
=============================================

1. TASK-LEVEL RETRIES:
   In a Databricks Job, each task can be configured with retry settings:

   Job JSON (task level):
   {
     "task_key": "ingest_orders",
     "notebook_task": {"notebook_path": "/pipelines/ingest_orders"},
     "max_retries": 3,
     "min_retry_interval_millis": 30000,   // 30 sec between retries
     "retry_on_timeout": true,
     "timeout_seconds": 3600
   }

   - max_retries: Number of retry attempts (0 = no retries, 3 = try up to 3 more times).
   - min_retry_interval_millis: Exponential backoff base (30s -> 60s -> 120s).
   - retry_on_timeout: Retry if the task times out (not just exceptions).
   - Best for: transient failures (network, S3 throttling, temporary OOM).
   - NOT for: logic errors that will fail every time.

2. REPAIR RUNS (Job-Level):
   When a multi-task Databricks Job fails (e.g., task 3 of 5 fails), you DON'T need
   to re-run the entire job. Use Repair Run to re-run ONLY failed tasks.

   CLI:
   databricks jobs repair-run --run-id <run_id> --rerun-tasks task_3,task_4

   API:
   POST /api/2.1/jobs/runs/repair
   {"run_id": 12345, "rerun_tasks": ["transform", "load"]}

   Flow:
   - Job has 5 tasks: extract -> validate -> transform -> load -> notify
   - 'transform' fails. 'load' and 'notify' are skipped (dependency not met).
   - Repair Run re-runs only 'transform' (and downstream tasks that depend on it).
   - 'extract' and 'validate' are NOT re-run (already succeeded).
   - This saves compute cost and time for long-running upstream tasks.

3. CHECKPOINT-BASED RECOVERY (Streaming):
   Structured Streaming writes offsets to a checkpoint location.
   If the query crashes, restarting with the same checkpoint resumes from the
   last committed offset -- no data loss, no duplicates.

   stream = (spark.readStream
       .format("delta")
       .load("/path/to/source")
       .writeStream
       .format("delta")
       .option("checkpointLocation", "/path/to/checkpoint")
       .toTable("target_table"))

   Key points:
   - checkpointLocation stores: offsets, state, committed logs.
   - On restart: driver reads last committed offset, seeks to that position.
   - Idempotent sinks (Delta MERGE) ensure no duplicates on replay.
   - NEVER delete checkpoint directory unless you want to start from scratch.

4. IDEMPOTENT RE-RUNS (MERGE with run_id):
   Pattern: tag each batch with a run_id. MERGE only if run_id hasn't been processed.

   MERGE INTO target AS t
   USING source AS s
   ON t.business_key = s.business_key AND t.run_id = s.run_id
   WHEN NOT MATCHED THEN INSERT *

   - If a task fails and is re-run with the SAME run_id, the MERGE is a no-op.
   - If a new run has a DIFFERENT run_id, new rows are inserted.
   - This ensures exactly-once semantics even with retries.

WHEN TO USE EACH:
   - Task retries: transient failures (network, OOM, throttling).
   - Repair Runs: multi-task job where upstream succeeded but mid-stream failed.
   - Checkpoint recovery: streaming pipelines that crash and need to resume.
   - Idempotent MERGE: any pipeline that might be re-run (batch or streaming).
""")
print("Q10 PASSED")

## Q11: Delta Lake Time Travel — Data Recovery & Rollback [ADVANCED]

**Category: ETL — Failures, Late Data, Schema Handling, Fault Tolerance**

A bad batch was written to your Delta table, corrupting 50K rows. Use **Delta Lake time travel** to recover:

1. Create a Delta table with good data and note the initial version
2. Simulate a bad write (overwrite with wrong data)
3. Use `DESCRIBE HISTORY` to list all versions and operations
4. Use `RESTORE TABLE TO VERSION` to roll back to the good version
5. Verify the data is restored correctly
6. Explain `VACUUM` for cleanup and its retention period implications

**Requirements:**
- Use Databricks SQL and Delta Lake commands
- Show both SQL (`RESTORE TABLE`) and Python (`df.option("versionAsOf", N)`) approaches
- Explain the difference between `versionAsOf` and `timestampAsOf`

In [0]:
from pyspark.sql import functions as F

# --- 1. Create table with good data ---
tbl = "workspace.default.day11_q11_timetravel"

good_data = [(i, f"customer_{i}", 100.0 * i) for i in range(1, 6)]
df_good = spark.createDataFrame(good_data, ["id", "name", "amount"])
df_good.write.mode("overwrite").saveAsTable(tbl)

print("=== Version 0: Good Data ===")
spark.table(tbl).show()

# --- 2. Simulate bad write (overwrite with wrong data) ---
bad_data = [(i, f"HACKED_{i}", -999.0) for i in range(1, 6)]
df_bad = spark.createDataFrame(bad_data, ["id", "name", "amount"])
df_bad.write.mode("overwrite").saveAsTable(tbl)

print("=== Version 1: Bad Data (simulated corruption) ===")
spark.table(tbl).show()

# --- 3. DESCRIBE HISTORY ---
history = spark.sql(f"DESCRIBE HISTORY {tbl}")
print("=== Table History ===")
history.select("version", "operation", "operationParameters").show(truncate=False)

# --- 4. RESTORE to version 0 (good data) ---
spark.sql(f"RESTORE TABLE {tbl} TO VERSION 0")
print("=== After RESTORE TO VERSION 0 ===")
spark.table(tbl).show()

# --- 5. Verify restoration ---
restored = spark.table(tbl).collect()
assert len(restored) == 5
assert restored[0]["name"] == "customer_1"
assert restored[0]["amount"] == 100.0
assert all(r["amount"] > 0 for r in restored)

# --- 6. Time travel read using versionAsOf ---
bad_version = spark.read.option("versionAsOf", 1).table(tbl)
print("=== Time Travel: Reading Version 1 (bad) ===")
bad_version.show()

# --- 7. TimestampAsOf approach ---
# You can also travel by timestamp:
# spark.read.option("timestampAsOf", "2025-10-23 10:00:00").table(tbl)

print("""
DELTA TIME TRAVEL EXPLAINED:
=============================

1. DESCRIBE HISTORY: Shows all operations (WRITE, RESTORE, DELETE, MERGE, UPDATE)
   with version numbers, timestamps, and operation parameters.

2. versionAsOf: Read table at a specific version number.
   spark.read.option("versionAsOf", 0).table("my_table")

3. timestampAsOf: Read table as of a specific timestamp.
   spark.read.option("timestampAsOf", "2025-10-23 10:00:00").table("my_table")

4. RESTORE TABLE TO VERSION: Physically rolls back the table data.
   Does NOT delete the log -- creates a new version with the restored state.
   The old versions are still in the log until VACUUM.

5. VACUUM:
   Removes old files no longer referenced by the Delta log.
   Default retention: 7 days (delta.deletedFileRetentionDuration).
   VACUUM tbl RETAIN 168 HOURS  -- keep files from last 7 days
   VACUUM tbl RETAIN 0 HOURS   -- remove all unreferenced files (DANGEROUS)
   After VACUUM: time travel to pre-VACUUM versions is NO LONGER POSSIBLE.

6. Best practice:
   - Use RESTORE for quick rollback of bad batches.
   - VACUUM after confirming the restore is correct.
   - Set retention based on your recovery window (7-30 days typical).
""")

print("Q11 PASSED")

## Q12: Junk & Degenerate Dimensions [INTERMEDIATE]

**Category: Data Modeling**

Explain and design **junk dimensions** and **degenerate dimensions**:

1. **Junk Dimension**: What is it? When do you combine multiple low-cardinality flags (yes/no, status codes) into a single dimension table instead of creating separate dimensions?
2. **Degenerate Dimension**: What is it? When does a dimension have no attributes of its own (e.g., transaction number) and is stored directly in the fact table?
3. Design a junk dimension table that combines `order_status`, `payment_method`, and `fulfillment_type` into a single dimension with a surrogate key
4. Design a fact table that includes a degenerate dimension (invoice_number)

**Requirements:**
- Explain the rationale for each pattern
- Provide table DDL with sample data
- Show how the junk dimension connects to the fact table via surrogate key

In [0]:
print("""
JUNK & DEGENERATE DIMENSIONS EXPLAINED:
=======================================

1. JUNK DIMENSION:
   - Combines multiple low-cardinality, unrelated flags/indicators into ONE dimension table.
   - Why: Creating a separate dimension for each 2-value flag (yes/no) is wasteful.
         A 5-flag row with 2 values each = 2^5 = 32 rows max in one junk dimension.
   - Each unique combination of flag values gets a surrogate key.
   - The fact table references this surrogate key instead of 5 separate FKs.

   Example: order_status (3 values) x payment_method (4 values) x fulfillment_type (3 values)
          = 36 possible combinations = 36-row junk dimension.

2. DEGENERATE DIMENSION:
   - A dimension key that has NO corresponding dimension table -- it's stored directly
     in the fact table as a column.
   - Most common: transaction/invoice/order numbers.
   - These have no attributes other than the ID itself, so a dimension table would just
     be a list of IDs with no useful attributes -- pointless.
   - The fact table stores the degenerate dimension as a column for drill-through/lookup.

   Example: invoice_number, transaction_id, order_number, ticket_number.
""")

# --- Junk Dimension Design ---
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Generate all combinations for the junk dimension
statuses = ["Pending", "Shipped", "Delivered"]
payments = ["Credit", "Debit", "PayPal", "Cash"]
fulfillments = ["Standard", "Express", "Same-Day"]

combos = []
for s in statuses:
    for p in payments:
        for f in fulfillments:
            combos.append((s, p, f))

junk_df = spark.createDataFrame(combos, ["order_status", "payment_method", "fulfillment_type"])

# Assign surrogate keys
junk_df = (junk_df
    .withColumn("junk_dim_key", F.row_number().over(Window.orderBy("order_status", "payment_method", "fulfillment_type")))
    .select("junk_dim_key", "order_status", "payment_method", "fulfillment_type"))

junk_tbl = "workspace.default.day11_q12_junk_dim"
junk_df.write.mode("overwrite").saveAsTable(junk_tbl)

print("=== Junk Dimension (order_status x payment_method x fulfillment_type) ===")
spark.table(junk_tbl).show(10, truncate=False)
print(f"Total rows: {spark.table(junk_tbl).count()} (3 x 4 x 3 = 36)")

# --- Fact Table with Degenerate Dimension ---
fact_data = [
    (1001, "INV-2025-001", 101, 5, 450.00, 1),   # junk_dim_key=1
    (1002, "INV-2025-002", 102, 3, 180.00, 2),   # junk_dim_key=2
    (1003, "INV-2025-003", 103, 7, 720.00, 15),   # junk_dim_key=15
    (1004, "INV-2025-004", 104, 2, 120.00, 28),  # junk_dim_key=28
]

fact_df = spark.createDataFrame(fact_data, ["order_id", "invoice_number", "customer_key", "qty", "amount", "junk_dim_key"])

fact_tbl = "workspace.default.day11_q12_fact_orders"
fact_df.write.mode("overwrite").saveAsTable(fact_tbl)

print("=== Fact Table (with degenerate dim: invoice_number, junk FK: junk_dim_key) ===")
spark.table(fact_tbl).show(truncate=False)

# --- Join fact to junk dimension ---
joined = (spark.table(fact_tbl)
    .join(spark.table(junk_tbl), on="junk_dim_key", how="left")
    .select("order_id", "invoice_number", "customer_key", "qty", "amount",
            "order_status", "payment_method", "fulfillment_type"))

print("=== Fact + Junk Dimension Join ===")
joined.show(truncate=False)

# --- Assertions ---
assert spark.table(junk_tbl).count() == 36  # 3 x 4 x 3 combinations
assert spark.table(fact_tbl).count() == 4
assert joined.count() == 4  # all joined successfully
assert "invoice_number" in [f.name for f in spark.table(fact_tbl).schema.fields]  # degenerate dim
print("Q12 PASSED")

## Q13: Surrogate Key Strategies with MERGE-based SCD Type 2 [ADVANCED]

**Category: Data Modeling**

Design and implement **surrogate key generation** strategies for Slowly Changing Dimension Type 2:

1. Compare three surrogate key strategies: **identity columns**, **hash keys** (SHA-256), and **monotonically increasing IDs**
2. Implement SCD Type 2 using **Delta MERGE** with **hash-based change detection** (hash_diff column)
3. When a customer's address changes: close the old record (`is_current = false`, set `end_date`) and insert a new record with a new surrogate key
4. Demonstrate the MERGE running twice (idempotent -- second run is a no-op)

**Requirements:**
- Use Delta Lake with UC managed tables
- Show the MERGE SQL with `WHEN MATCHED` (close old) and `WHEN NOT MATCHED` (insert new)
- Explain why hash-based change detection is more efficient than comparing every column
- Verify with sample data that address change creates a new SCD2 record

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, BooleanType
from datetime import date

# --- Surrogate Key Strategies Explained ---
print("""
SURROGATE KEY STRATEGIES:
=========================
1. IDENTITY COLUMN: Delta supports GENERATED ALWAYS AS IDENTITY.
   Pros: Simple, ordered, database-managed.
   Cons: Not deterministic (same data = different keys on re-run).

2. HASH KEY (SHA-256): hash the natural key (e.g., customer_id + valid_from).
   Pros: Deterministic, idempotent, parallel-safe.
   Cons: Large (64 hex chars), not human-readable.

3. MONOTONICALLY_INCREASING: F.monotonically_increasing_id().
   Pros: Fast, parallel.
   Cons: Not sequential, not stable across runs.

For SCD Type 2, HASH keys are preferred because they make MERGE idempotent.
""")

# --- Create target SCD2 table ---
target_tbl = "workspace.default.day11_q13_scd2"

target_schema = StructType([
    StructField("customer_sk", StringType(), True),       # hash-based surrogate key
    StructField("customer_id", IntegerType(), True),     # natural key
    StructField("name", StringType(), True),
    StructField("address", StringType(), True),
    StructField("hash_diff", StringType(), True),        # hash of attributes
    StructField("valid_from", DateType(), True),
    StructField("valid_to", DateType(), True),
    StructField("is_current", BooleanType(), True),
])

# Initial data: 2 customers
initial = [
    ("hash_1_2025", 1, "Alice", "123 Main St", None, date(2025, 1, 1), None, True),
    ("hash_2_2025", 2, "Bob",   "456 Oak Ave", None, date(2025, 1, 1), None, True),
]

# Recompute hash_diff for initial data
df_init = spark.createDataFrame(initial, target_schema)
df_init = df_init.withColumn("hash_diff", F.sha2(F.concat(F.col("name"), F.lit("|"), F.col("address")), 256))
df_init.write.mode("overwrite").saveAsTable(target_tbl)

print("=== Initial SCD2 Table ===")
spark.table(target_tbl).show(truncate=False)

# --- Source: Alice's address changed ---
source_data = [
    (1, "Alice", "789 Pine Rd", date(2025, 6, 1)),
    (2, "Bob",   "456 Oak Ave", date(2025, 6, 1)),  # unchanged
]
source_df = spark.createDataFrame(source_data, ["customer_id", "name", "address", "effective_date"])
source_df = source_df.withColumn("hash_diff", F.sha2(F.concat(F.col("name"), F.lit("|"), F.col("address")), 256))
source_df = source_df.withColumn("customer_sk", F.sha2(F.concat(F.col("customer_id"), F.lit("|"), F.col("effective_date")), 256))
source_df.createOrReplaceTempView("source_scd2")

# --- MERGE: SCD Type 2 ---
spark.sql(f"""
MERGE INTO {target_tbl} AS t
USING source_scd2 AS s
ON t.customer_id = s.customer_id AND t.is_current = true
WHEN MATCHED AND t.hash_diff != s.hash_diff
  THEN UPDATE SET
    t.is_current = false,
    t.valid_to = s.effective_date
""")

# Insert new current records for changed customers
spark.sql(f"""
INSERT INTO {target_tbl}
SELECT s.customer_sk, s.customer_id, s.name, s.address, s.hash_diff,
       s.effective_date, NULL, true
FROM source_scd2 s
JOIN {target_tbl} t ON t.customer_id = s.customer_id AND t.is_current = false
WHERE t.valid_to = s.effective_date AND t.hash_diff != s.hash_diff
""")

print("=== After SCD2 MERGE (Alice's address changed) ===")
spark.table(target_tbl).orderBy("customer_id", "valid_from").show(truncate=False)

# --- Idempotent: Run MERGE again (should be no-op) ---
spark.sql(f"""
MERGE INTO {target_tbl} AS t
USING source_scd2 AS s
ON t.customer_id = s.customer_id AND t.is_current = true
WHEN MATCHED AND t.hash_diff != s.hash_diff
  THEN UPDATE SET
    t.is_current = false,
    t.valid_to = s.effective_date
""")

spark.sql(f"""
INSERT INTO {target_tbl}
SELECT s.customer_sk, s.customer_id, s.name, s.address, s.hash_diff,
       s.effective_date, NULL, true
FROM source_scd2 s
JOIN {target_tbl} t ON t.customer_id = s.customer_id AND t.is_current = false
WHERE t.valid_to = s.effective_date AND t.hash_diff != s.hash_diff
""")

print("=== After Second MERGE (should be no-op - idempotent) ===")
result = spark.table(target_tbl).orderBy("customer_id", "valid_from")
result.show(truncate=False)

# --- Assertions ---
assert result.count() == 3  # 2 initial + 1 new for Alice
alice = result.filter(F.col("customer_id") == 1).collect()
assert len(alice) == 2  # old (closed) + new (current)
assert alice[0]["is_current"] == False  # old record closed
assert alice[1]["is_current"] == True   # new record current
assert alice[1]["address"] == "789 Pine Rd"
print("Q13 PASSED")

## Q14: Snowpipe vs Snowpipe Streaming — Continuous Ingestion [INTERMEDIATE]

**Category: Snowflake**

Compare **Snowpipe** and **Snowpipe Streaming** for continuous data ingestion into Snowflake:

1. What is **Snowpipe**? How does it work with cloud storage event notifications?
2. What is **Snowpipe Streaming**? How does it use the REST API for direct row insertion?
3. Compare **latency**, **throughput**, **cost model**, and **use cases** for each
4. Provide SQL examples for creating a pipe and ingesting data
5. When would you choose one over the other?

**Requirements:**
- Explain the architecture of each approach
- Provide CREATE PIPE and COPY INTO SQL examples
- Discuss the trade-offs in an interview-ready format

In [0]:
print("""
SNOWPIPE vs SNOWPIPE STREAMING:
===============================

1. SNOWPIPE (File-Based Auto-Ingest):
   Architecture:
   [Cloud Storage (S3/GCS/Azure)] --event notification--> [Snowpipe] --> [Snowflake Table]

   - Cloud storage sends event notifications (S3 SQS, GCS Pub/Sub, Azure Event Grid)
   - Snowpipe detects new files, reads them, and COPY INTO the target table.
   - Files must be in supported formats (CSV, JSON, Parquet, Avro).

   SQL Example:
   CREATE PIPE my_pipe
   AUTO_INTEGRATE = true
   AS
   COPY INTO my_table
   FROM @my_stage
   FILE_FORMAT = (TYPE = CSV SKIP_HEADER = 1);

   -- Check pipe status
   SELECT * FROM TABLE(SYSTEM$PIPE_STATUS('my_pipe'));

   - Latency: 10-60 seconds (depends on notification + queue processing).
   - Throughput: Limited by file size and COPY throughput.
   - Cost: Charged per compute second used by the Snowpipe warehouse.

2. SNOWPIPE STREAMING (REST API Direct Insert):
   Architecture:
   [Application SDK] --REST API--> [Snowpipe Streaming] --> [Snowflake Table]

   - No file staging required -- rows are sent directly via REST API.
   - Uses Snowflake Ingest SDK (Java/Python) for streaming inserts.
   - Data is buffered client-side and flushed in batches.

   Python SDK Example:
   from snowflake.ingest import StreamingIngestClient
   client = StreamingIngestClient('my_client', account='myaccount')
   channel = client.open_channel('my_channel', 'my_db.my_schema.my_table')
   channel.insert_rows([{'id': 1, 'name': 'Alice'}])
   channel.close()

   - Latency: Sub-second to few seconds (near real-time).
   - Throughput: High -- parallel channels, batched inserts.
   - Cost: Charged per compute second, but more efficient for frequent small batches.

3. COMPARISON TABLE:
   Feature           | Snowpipe                | Snowpipe Streaming
   ------------------|------------------------|----------------------
   Input             | Files in cloud storage  | REST API (rows)
   Latency           | 10-60 seconds           | < 1 second
   Staging           | Required (S3/GCS/Azure)  | Not required
   File formats      | CSV, JSON, Parquet, etc | Schema-defined rows
   Use case          | Batch/micro-batch files | Real-time streaming
   SDK               | SQL only                | Java/Python SDK
   Cost              | Per-file COPY compute   | Per-channel compute
   Replaces          | Bulk COPY scheduling    | Kafka/Kinesis connectors

4. WHEN TO CHOOSE:
   - Snowpipe: You already have files landing in S3/GCS and want automatic ingestion.
     Best for micro-batch (files every few minutes).
   - Snowpipe Streaming: You need sub-second latency and don't want to write files.
     Best for IoT, clickstream, real-time event streaming.
   - Hybrid: Use Snowpipe for file-based historical loads and Streaming for real-time.

INTERVIEW TALKING POINTS:
   - Snowpipe uses serverless compute (no warehouse needed for the COPY itself).
   - Snowpipe Streaming can achieve sub-second latency but requires SDK integration.
   - Both support exactly-once delivery (no duplicates on retry).
   - Snowpipe is fire-and-forget; Streaming requires connection management.
""")
print("Q14 PASSED")

## Q15: Snowflake Task DAG with Serverless Tasks — Orchestration [ADVANCED]

**Category: Snowflake**

Design a **Snowflake Task DAG** for an ETL pipeline using **serverless tasks**:

1. Create a task DAG with 4 tasks: `extract_raw` -> `transform_clean` -> `load_dims` -> `load_facts`
2. Use **serverless compute** (no warehouse needed) for each task
3. Configure task dependencies (`AFTER` clause) and schedule the root task
4. Add **error handling**: `WHEN ERROR THEN` clause, task failure notifications via integration
5. Explain how serverless tasks differ from user-managed warehouse tasks
6. Compare Snowflake task DAGs with Databricks Jobs orchestration

**Requirements:**
- Provide CREATE TASK SQL for each task in the DAG
- Show how to set up the root schedule and child dependencies
- Explain serverless task billing model

In [0]:
print("""
SNOWFLAKE TASK DAG WITH SERVERLESS TASKS:
=========================================

1. TASK DAG ARCHITECTURE:

   [Root: extract_raw] (scheduled every 1 hour)
     |
     v
   [Child: transform_clean] (AFTER extract_raw)
     |
     v
   [Child: load_dims] (AFTER transform_clean)
     |
     v
   [Child: load_facts] (AFTER load_dims)

2. SQL: CREATE TASK STATEMENTS

   -- Root task: runs every hour using serverless compute
   CREATE OR REPLACE TASK extract_raw
     WAREHOUSE = SIZE = 'XSMALL'  -- OR omit warehouse for serverless
     SCHEDULE = '60 MINUTE'
   AS
     COPY INTO raw_orders FROM @s3_stage/orders/
     FILE_FORMAT = (TYPE = PARQUET);

   -- Serverless task (no warehouse specified)
   CREATE OR REPLACE TASK transform_clean
     AFTER extract_raw
   AS
     INSERT INTO clean_orders
     SELECT id, UPPER(name), amount, TRY_CAST(date AS DATE)
     FROM raw_orders WHERE amount > 0;

   -- Child task: load dimensions
   CREATE OR REPLACE TASK load_dims
     AFTER transform_clean
   AS
     MERGE INTO dim_customer AS t
     USING (SELECT DISTINCT customer_id, customer_name FROM clean_orders) AS s
     ON t.customer_id = s.customer_id
     WHEN NOT MATCHED THEN INSERT (customer_id, customer_name)
     VALUES (s.customer_id, s.customer_name);

   -- Child task: load facts
   CREATE OR REPLACE TASK load_facts
     AFTER load_dims
   AS
     INSERT INTO fact_orders
     SELECT c.id, d.customer_sk, c.amount, c.date
     FROM clean_orders c JOIN dim_customer d ON c.customer_id = d.customer_id;

   -- Resume all tasks (tasks are suspended by default)
   ALTER TASK extract_raw RESUME;

3. SERVERLESS TASKS:
   - If WAREHOUSE is NOT specified, Snowflake uses serverless compute.
   - Snowflake auto-scales the compute based on task complexity.
   - Billing: per-second, only while the task SQL is executing.
   - No idle warehouse costs -- you don't pay for a warehouse sitting idle between runs.
   - Contrast: user-managed warehouse tasks require a running warehouse (or auto-suspend).

4. ERROR HANDLING:
   -- Task with error notification
   CREATE OR REPLACE TASK extract_raw
     SCHEDULE = '60 MINUTE'
     WHEN SYSTEM$STREAM_HAS_DATA('raw_orders_stream')  -- conditional run
   AS
     BEGIN
       COPY INTO raw_orders FROM @s3_stage/orders/;
     EXCEPTION
       WHEN OTHER THEN
         INSERT INTO task_error_log VALUES (CURRENT_TIMESTAMP(), SQLERRM);
         -- Send notification via notification integration
         CALL SYSTEM$SEND_SNS_NOTIFICATION('pipeline-alerts', SQLERRM);
     END;

   -- Monitor task history
   SELECT * FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(
     TASK_NAME => 'extract_raw',
     SCHEDULED_TIME_RANGE_START => DATEADD('hour', -24, CURRENT_TIMESTAMP())
   ));

5. SNOWFLAKE TASK DAG vs DATABRICKS JOBS:
   Feature        | Snowflake Tasks           | Databricks Jobs
   ---------------|--------------------------|--------------------------
   Dependencies   | AFTER clause             | Run If / depends_on
   Schedule      | SCHEDULE (cron or interval)| cron / file arrival / table update
   Compute       | Serverless or WH         | Cluster or serverless
   Error handling| EXCEPTION block in SQL   | max_retries + alert on fail
   Retry         | No built-in retry        | max_retries per task
   Monitoring    | TASK_HISTORY view        | system.lakeflow.* tables
   Language      | SQL only                 | Python, SQL, Scala, notebooks
   DAG depth     | Up to 3 levels (root+2)  | Unlimited depth
   Notifications | SNS/Email integration    | Email, webhook, Slack

   Key difference: Snowflake tasks are SQL-only; Databricks Jobs support any language.
   Snowflake limits DAG depth to 3; Databricks has no depth limit.
   Databricks has built-in retry; Snowflake needs custom error handling.
""")
print("Q15 PASSED")

## Q16: Jobs Retry & Timeout Configuration — Resilient Pipelines [INTERMEDIATE]

**Category: Monitoring, Logging & Orchestration**

Design a **resilient Databricks Job** with proper retry and timeout configuration:

1. Configure **task-level retries**: `max_retries`, `min_retry_interval_millis`, `retry_on_timeout`
2. Configure **job-level retry on failure**: `retry_on_failure` with `max_retries`
3. Explain the **Run If** dependency conditions: `all_success`, `all_done`, `none_failed`, `at_least_one`, `all_failed`
4. Set appropriate **timeout_seconds** for each task type (notebook, SQL, pipeline)
5. Design a 4-task pipeline with different retry strategies for each task

**Requirements:**
- Provide a Databricks Job JSON config snippet
- Explain when to use `all_done` vs `all_success` for dependencies
- Discuss exponential backoff behavior in `min_retry_interval_millis`
- Explain how `retry_on_timeout` differs from `retry_on_failure`

In [0]:
print("""
DATABRICKS JOBS RETRY & TIMEOUT CONFIGURATION:
===============================================

1. TASK-LEVEL RETRY CONFIG:
   Each task in a Databricks Job can have independent retry settings:

   {
     "task_key": "ingest_data",
     "notebook_task": {"notebook_path": "/pipelines/ingest"},
     "max_retries": 3,
     "min_retry_interval_millis": 30000,
     "retry_on_timeout": true,
     "timeout_seconds": 1800
   }

   - max_retries: Number of ADDITIONAL attempts (3 = try 4 times total).
   - min_retry_interval_millis: Base delay between retries (exponential backoff).
     30s -> 60s -> 120s (doubles each retry).
   - retry_on_timeout: If true, retry when the task TIMES OUT (not just errors).
     If false, timeout = immediate failure, no retry.
   - timeout_seconds: Hard limit for the task. 0 = no timeout.

2. JOB-LEVEL RETRY ON FAILURE:
   {
     "name": "etl_pipeline",
     "retry_on_failure": {"max_retries": 2},
     "tasks": [...]
   }

   - retry_on_failure: Retries the ENTIRE job if any task fails (after task retries exhausted).
   - This is a fallback: task-level retries run first, then job-level.
   - Use case: pipeline that depends on external systems -- if all tasks fail,
     retry the whole job (maybe the external system recovered).

3. RUN IF DEPENDENCY CONDITIONS:
   When task B depends on task A, you can control WHEN B runs based on A's outcome:

   all_success (DEFAULT): B runs only if A AND all its dependencies succeeded.
   all_done:     B runs when A finishes, regardless of success/failure. (cleanup task)
   none_failed:  B runs if A succeeded OR was skipped, but NOT if A failed.
   at_least_one: B runs if at least one of A's branches succeeded. (fan-out + fan-in)
   all_failed:   B runs only if ALL of A's branches failed. (escalation/alerting)

   When to use each:
   - all_success: Normal sequential pipeline.
   - all_done: Cleanup/notification tasks that must run no matter what.
   - none_failed: Conditional downstream that should skip on failure but not error.
   - at_least_one: Fan-out to multiple workers, continue if at least one succeeded.

4. COMPLETE 4-TASK PIPELINE EXAMPLE:

   {
     "name": "resilient_etl",
     "retry_on_failure": {"max_retries": 1},
     "tasks": [
       {
         "task_key": "extract",
         "notebook_task": {"notebook_path": "/pipelines/extract"},
         "max_retries": 3,
         "min_retry_interval_millis": 30000,
         "retry_on_timeout": true,
         "timeout_seconds": 1800
       },
       {
         "task_key": "transform",
         "depends_on": [{"task_key": "extract", "condition": "all_success"}],
         "notebook_task": {"notebook_path": "/pipelines/transform"},
         "max_retries": 2,
         "min_retry_interval_millis": 60000,
         "retry_on_timeout": false,
         "timeout_seconds": 3600
       },
       {
         "task_key": "load",
         "depends_on": [{"task_key": "transform", "condition": "all_success"}],
         "notebook_task": {"notebook_path": "/pipelines/load"},
         "max_retries": 1,
         "min_retry_interval_millis": 120000,
         "retry_on_timeout": true,
         "timeout_seconds": 900
       },
       {
         "task_key": "notify_cleanup",
         "depends_on": [{"task_key": "load", "condition": "all_done"}],
         "notebook_task": {"notebook_path": "/pipelines/cleanup"},
         "max_retries": 0,
         "timeout_seconds": 300
       }
     ],
     "trigger": {"file_arrival": {"url": "s3://my-bucket/trigger.flag"}}
   }

   Design rationale:
   - extract: 3 retries (network failures common when pulling from external API).
   - transform: 2 retries, no timeout retry (logic errors won't fix themselves).
   - load: 1 retry, timeout retry on (DB writes can timeout under load).
   - notify_cleanup: 0 retries, all_done (must run regardless of pipeline success).

5. RETRY BEHAVIOR SUMMARY:
   - Task fails -> task retries (up to max_retries) with exponential backoff.
   - All task retries exhausted -> task is marked failed.
   - If job has retry_on_failure -> entire job retries (up to max_retries).
   - During Repair Run: only failed tasks + their downstream are re-run.
""")
print("Q16 PASSED")

## Q17: Unified Observability with System Tables [ADVANCED]

**Category: Monitoring, Logging & Orchestration**

Build a **unified observability layer** using Databricks system tables to monitor cost, performance, and security:

1. **Cost monitoring**: Use `system.billing.usage` to track daily spend by SKU, workspace, and tag
2. **Performance monitoring**: Use `system.query.history` to find slow queries, high scan queries, and query trends
3. **Security monitoring**: Use `system.access.audit` to track sensitive table access and failed authentication
4. Build a **unified dashboard query** that joins cost + performance metrics
5. Set up **SQL alerts** for anomaly detection (spend > threshold, query duration spike)

**Requirements:**
- Provide SQL queries for each system table
- Guard against schema differences with try/except
- Explain which system tables are available and their key columns
- Show how to set up a SQL alert for cost threshold breach

In [0]:
print("""
UNIFIED OBSERVABILITY WITH DATABRICKS SYSTEM TABLES:
=====================================================

Available System Tables (key ones):
  system.billing.usage          -- cost by SKU, workspace, tags
  system.billing.listable       -- all billable SKUs and rates
  system.query.history          -- all SQL queries (duration, bytes scanned, user)
  system.access.audit           -- audit log (who accessed what, when)
  system.access.table_lineage   -- table-level lineage
  system.lakeflow.job_run_timeline -- Job runs and status
  system.information_schema.*   -- metadata about catalogs, tables, columns
""")

# --- 1. COST MONITORING: system.billing.usage ---
try:
    billing_df = spark.sql("""
    SELECT
        usage_date,
        sku_name,
        SUM(usage_quantity) AS total_usage,
    SUM(usage_quantity * pricing.pricing_default) AS estimated_cost
    FROM system.billing.usage
    WHERE usage_date >= DATEADD(day, -7, CURRENT_DATE())
    GROUP BY usage_date, sku_name
    ORDER BY usage_date DESC, estimated_cost DESC
    LIMIT 10
    """)
    print("=== Cost by SKU (Last 7 Days) ===")
    billing_df.show(truncate=False)
except Exception as e:
    print(f"system.billing.usage query failed (schema may differ): {e}")

# --- 2. PERFORMANCE MONITORING: system.query.history ---
try:
    slow_queries = spark.sql("""
    SELECT
        statement_id,
        statement_text,
        total_task_time_ms / 1000.0 AS duration_seconds,
        read_bytes / 1024 / 1024 / 1024 AS read_gb,
        start_time
    FROM system.query.history
    WHERE start_time >= DATEADD(hour, -24, CURRENT_TIMESTAMP())
      AND total_task_time_ms > 60000
    ORDER BY total_task_time_ms DESC
    LIMIT 10
    """)
    print("=== Slow Queries (>60s, Last 24h) ===")
    slow_queries.show(truncate=True)
except Exception as e:
    print(f"system.query.history query failed: {e}")

# --- 3. SECURITY MONITORING: system.access.audit ---
try:
    audit_df = spark.sql("""
    SELECT
        event_time,
        action_name,
        actor_name,
        request_params,
        source_ip_address
    FROM system.access.audit
    WHERE event_time >= DATEADD(hour, -24, CURRENT_TIMESTAMP())
      AND action_name IN ('getTable', 'createTable', 'dropTable')
    ORDER BY event_time DESC
    LIMIT 10
    """)
    print("=== Recent Table Access Events ===")
    audit_df.show(truncate=True)
except Exception as e:
    print(f"system.access.audit query failed: {e}")

# --- 4. UNIFIED DASHBOARD QUERY ---
print("""
UNIFIED OBSERVABILITY QUERY (pseudo-code):

WITH cost AS (
  SELECT usage_date, SUM(usage_quantity * pricing_default) AS daily_cost
  FROM system.billing.usage
  WHERE usage_date >= DATEADD(day, -7, CURRENT_DATE())
  GROUP BY usage_date
),
performance AS (
  SELECT DATE(start_time) AS query_date,
         COUNT(*) AS query_count,
         AVG(total_task_time_ms) AS avg_duration_ms
  FROM system.query.history
  WHERE start_time >= DATEADD(day, -7, CURRENT_DATE())
  GROUP BY DATE(start_time)
),
security AS (
  SELECT DATE(event_time) AS audit_date,
         COUNT(*) AS access_events
  FROM system.access.audit
  WHERE event_time >= DATEADD(day, -7, CURRENT_DATE())
  GROUP BY DATE(event_time)
)
SELECT c.usage_date, c.daily_cost, p.query_count, p.avg_duration_ms, s.access_events
FROM cost c
LEFT JOIN performance p ON c.usage_date = p.query_date
LEFT JOIN security s ON c.usage_date = s.audit_date
ORDER BY c.usage_date DESC;
""")

# --- 5. SQL ALERT FOR COST THRESHOLD ---
print("""
SQL ALERT CONFIG:
  Query: SELECT SUM(usage_quantity * pricing_default) AS daily_cost
         FROM system.billing.usage
         WHERE usage_date = CURRENT_DATE() - 1
  Alert: daily_cost > 1000  (trigger when daily spend exceeds $1000)
  Schedule: Daily at 8 AM (after previous day's billing is complete)
  Notification: Email + Webhook to Slack #data-alerts
""")
print("Q17 PASSED")

## Q18: Self-Healing Pipeline — Failure Detection + Retry + DLQ + Alerting [ADVANCED]

**Category: Monitoring, Logging & Orchestration**

Design a **self-healing batch processing pipeline** that automatically detects failures, retries, quarantines bad records, and sends alerts:

1. **Failure detection**: Wrap each processing step in try/except with structured error capture
2. **Automatic retry**: Implement exponential backoff retry logic for transient failures
3. **Dead letter queue**: Route records that fail validation after all retries to a DLQ table
4. **Alerting**: Send a notification (email/webhook) when failures exceed a threshold
5. **Circuit breaker**: Stop processing if failure rate exceeds 50% (prevent cascading failures)

**Requirements:**
- Implement a Python class `SelfHealingPipeline` with methods: `process_batch`, `retry_with_backoff`, `quarantine`, `send_alert`
- Demonstrate with sample data that includes both good and bad records
- Show the circuit breaker in action (open after threshold)
- Include logging at each step for auditability

In [0]:
import time
import logging
from dataclasses import dataclass, field
from typing import List, Optional

# --- Logging Setup ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger("SelfHealingPipeline")

@dataclass
class ProcessingResult:
    success_count: int = 0
    failed_count: int = 0
    quarantined: List[dict] = field(default_factory=list)

class CircuitBreaker:
    """Opens when failure rate exceeds threshold."""
    def __init__(self, failure_threshold=0.5, min_requests=10):
        self.failure_threshold = failure_threshold
        self.min_requests = min_requests
        self.failure_count = 0
        self.success_count = 0
        self.is_open = False

    def record_success(self):
        self.success_count += 1
        self._check_state()

    def record_failure(self):
        self.failure_count += 1
        self._check_state()

    def _check_state(self):
        total = self.success_count + self.failure_count
        if total >= self.min_requests:
            rate = self.failure_count / total
            if rate >= self.failure_threshold:
                self.is_open = True
                logger.warning(f"Circuit breaker OPEN (failure rate: {rate:.1%})")

class SelfHealingPipeline:
    def __init__(self, max_retries=3, base_delay=1, failure_threshold=0.5):
        self.max_retries = max_retries
        self.base_delay = base_delay
        self.breaker = CircuitBreaker(failure_threshold=failure_threshold, min_requests=10)
        self.dlq: List[dict] = []
        self.alerts: List[str] = []

    def retry_with_backoff(self, func, record):
        """Retry a function with exponential backoff."""
        for attempt in range(self.max_retries + 1):
            try:
                return func(record), True
            except Exception as e:
                if attempt < self.max_retries:
                    delay = self.base_delay * (2 ** attempt)
                    logger.warning(f"Retry {attempt+1}/{self.max_retries} for record {record.get('id')}: {e}. Waiting {delay}s")
                    time.sleep(delay)
                else:
                    logger.error(f"All retries exhausted for record {record.get('id')}: {e}")
                    return None, False
        return None, False

    def validate_record(self, record):
        """Business validation: raises ValueError if invalid."""
        if record.get('id') is None:
            raise ValueError("Missing id")
        if record.get('amount', 0) <= 0:
            raise ValueError(f"Invalid amount: {record.get('amount')}")
        if record.get('customer') is None:
            raise ValueError("Missing customer")
        return record

    def quarantine(self, record, error_reason):
        """Send bad record to DLQ."""
        dlq_entry = {**record, 'error_reason': error_reason, 'quarantined_at': time.time()}
        self.dlq.append(dlq_entry)
        logger.info(f"Quarantined record {record.get('id')}: {error_reason}")

    def send_alert(self, message):
        """Send alert (simulated -- in production: webhook/email)."""
        self.alerts.append(message)
        logger.warning(f"ALERT: {message}")

    def process_batch(self, records) -> ProcessingResult:
        result = ProcessingResult()

        if self.breaker.is_open:
            self.send_alert("Circuit breaker OPEN -- batch skipped")
            return result

        for record in records:
            if self.breaker.is_open:
                self.send_alert("Circuit breaker opened mid-batch -- stopping")
                break

            processed, success = self.retry_with_backoff(self.validate_record, record)
            if success:
                result.success_count += 1
                self.breaker.record_success()
                logger.info(f"Processed record {record.get('id')} OK")
            else:
                result.failed_count += 1
                self.breaker.record_failure()
                self.quarantine(record, "validation_failed_after_retries")

        # Alert if failure rate is high
        total = result.success_count + result.failed_count
        if total > 0 and result.failed_count / total > 0.3:
            self.send_alert(f"High failure rate: {result.failed_count}/{total} records failed")

        return result

# --- Demo with sample data ---
records = [
    {"id": 1, "customer": "Alice", "amount": 100.0},   # good
    {"id": 2, "customer": "Bob",   "amount": -50.0},    # bad: negative
    {"id": 3, "customer": "Charlie", "amount": 200.0},  # good
    {"id": None, "customer": "Diana", "amount": 150.0}, # bad: null id
    {"id": 4, "customer": None, "amount": 300.0},       # bad: null customer
    {"id": 5, "customer": "Eve",  "amount": 75.0},     # good
    {"id": 6, "customer": "Frank", "amount": 500.0},   # good
    {"id": 7, "customer": "Grace", "amount": 0.0},    # bad: zero amount
    {"id": 8, "customer": "Heidi", "amount": 250.0},   # good
    {"id": 9, "customer": "Ivan",  "amount": 180.0},   # good
]

pipeline = SelfHealingPipeline(max_retries=2, base_delay=0.1, failure_threshold=0.4)
result = pipeline.process_batch(records)

print(f"\n=== Results ===")
print(f"Success: {result.success_count}")
print(f"Failed:  {result.failed_count}")
print(f"DLQ entries: {len(pipeline.dlq)}")
print(f"Alerts: {pipeline.alerts}")
print(f"Circuit breaker open: {pipeline.breaker.is_open}")

# --- Assertions ---
assert result.success_count == 6   # records 1,3,5,6,8,9
assert result.failed_count == 4     # records 2, null_id, null_customer, 7
assert len(pipeline.dlq) == 4
assert len(pipeline.alerts) >= 1   # at least one alert for high failure rate
print("Q18 PASSED")

---

## Score Card — Day 11

| # | Category | Topic | Difficulty | Result |
|---|----------|-------|------------|--------|
| Q1 | SQL Advanced | CUME_DIST / PERCENT_RANK percentile analysis | INTERMEDIATE | ⬜ |
| Q2 | SQL Advanced | Conditional aggregation pivot + YoY growth | ADVANCED | ⬜ |
| Q3 | Python | Trapping Rain Water (two-pointer) | INTERMEDIATE | ⬜ |
| Q4 | Python | Trie with autocomplete & prefix search | ADVANCED | ⬜ |
| Q5 | PySpark Coding | Multi-level groupBy + top-N window ranking | INTERMEDIATE | ⬜ |
| Q6 | PySpark Coding | Schema evolution with mergeSchema & nested structs | ADVANCED | ⬜ |
| Q7 | Spark Optimization | Shuffle partitioning & tuning internals | INTERMEDIATE | ⬜ |
| Q8 | Spark Optimization | Dynamic resource allocation & executor lifecycle | ADVANCED | ⬜ |
| Q9 | ETL Fault Tolerance | Dead letter queue: quarantine + replay | INTERMEDIATE | ⬜ |
| Q10 | ETL Fault Tolerance | Fault tolerance: retries + Repair Runs + checkpoint | ADVANCED | ⬜ |
| Q11 | ETL Fault Tolerance | Delta Lake time travel recovery & rollback | ADVANCED | ⬜ |
| Q12 | Data Modeling | Junk & degenerate dimensions | INTERMEDIATE | ⬜ |
| Q13 | Data Modeling | Surrogate key strategies with MERGE SCD Type 2 | ADVANCED | ⬜ |
| Q14 | Snowflake | Snowpipe vs Snowpipe Streaming ingestion | INTERMEDIATE | ⬜ |
| Q15 | Snowflake | Task DAG with serverless tasks orchestration | ADVANCED | ⬜ |
| Q16 | Monitoring & Orchestration | Jobs retry & timeout configuration | INTERMEDIATE | ⬜ |
| Q17 | Monitoring & Orchestration | Unified observability with system tables | ADVANCED | ⬜ |
| Q18 | Monitoring & Orchestration | Self-healing pipeline: detect + retry + DLQ + alert | ADVANCED | ⬜ |

**Summary:** 18 questions | 7 INTERMEDIATE | 11 ADVANCED | 8 categories

**Instructions:** Replace ⬜ with ✅ after solving each question. Time yourself: aim for 30-45 seconds per question in an interview setting.